###### 128_7_split_string_column_to_rows_practice

In [37]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [38]:
df = pd.DataFrame(
    [
         ["T001", "J69P", "MELMB", "コイル/Buck-IC", "RH Lo不灯", 5],
        ["T002", "J69P", "MELMB", "コイル", "RH Hi/Lo不灯", 4],
        ["T003", "J69P", "WSE", "Buck-IC/MCU", "全灯火不点灯", 5],
        ["T004", "310D", "GSE", "はんだ/コネクタ", "点灯不安定", 3],
        ["T005", "310D", "ISE", "異物", "点灯不安定", 2],
        ["T006", "J69P", "GSE", "コイル/異物", "RH Lo不灯", 4],
        ["T007", "310D", "WSE", "NTF", "再現せず", 1],
        ["T008", "J69P", "ISE", "コイル/Buck-IC/MCU", "RH Lo不灯", 5],
    ],
    columns=[
        "ticket_id",
        "project",
        "site",
        "確認部品",
        "symptom",
        "risk_score",
    ],
)
df.head()

,ticket_id,project,site,確認部品,symptom,risk_score
0,T001,J69P,MELMB,コイル/Buck-IC,RH Lo不灯,5
1,T002,J69P,MELMB,コイル,RH Hi/Lo不灯,4
2,T003,J69P,WSE,Buck-IC/MCU,全灯火不点灯,5
3,T004,310D,GSE,はんだ/コネクタ,点灯不安定,3
4,T005,310D,ISE,異物,点灯不安定,2


In [39]:
df["確認部品"] = df["確認部品"].str.split("/")

df

,ticket_id,project,site,確認部品,symptom,risk_score
0,T001,J69P,MELMB,"[コイル, Buck-IC]",RH Lo不灯,5
1,T002,J69P,MELMB,[コイル],RH Hi/Lo不灯,4
2,T003,J69P,WSE,"[Buck-IC, MCU]",全灯火不点灯,5
3,T004,310D,GSE,"[はんだ, コネクタ]",点灯不安定,3
4,T005,310D,ISE,[異物],点灯不安定,2
5,T006,J69P,GSE,"[コイル, 異物]",RH Lo不灯,4
6,T007,310D,WSE,[NTF],再現せず,1
7,T008,J69P,ISE,"[コイル, Buck-IC, MCU]",RH Lo不灯,5


In [40]:
df_long = df.explode("確認部品").reset_index(drop=True)
df_long

,ticket_id,project,site,確認部品,symptom,risk_score
0,T001,J69P,MELMB,コイル,RH Lo不灯,5
1,T001,J69P,MELMB,Buck-IC,RH Lo不灯,5
2,T002,J69P,MELMB,コイル,RH Hi/Lo不灯,4
3,T003,J69P,WSE,Buck-IC,全灯火不点灯,5
4,T003,J69P,WSE,MCU,全灯火不点灯,5
5,T004,310D,GSE,はんだ,点灯不安定,3
6,T004,310D,GSE,コネクタ,点灯不安定,3
7,T005,310D,ISE,異物,点灯不安定,2
8,T006,J69P,GSE,コイル,RH Lo不灯,4
9,T006,J69P,GSE,異物,RH Lo不灯,4


In [41]:
df_high_risk = df_long[
    df_long["risk_score"] >= 4
].copy()

In [42]:
df_high_risk.head(5)

,ticket_id,project,site,確認部品,symptom,risk_score
0,T001,J69P,MELMB,コイル,RH Lo不灯,5
1,T001,J69P,MELMB,Buck-IC,RH Lo不灯,5
2,T002,J69P,MELMB,コイル,RH Hi/Lo不灯,4
3,T003,J69P,WSE,Buck-IC,全灯火不点灯,5
4,T003,J69P,WSE,MCU,全灯火不点灯,5


In [43]:
df_parts_count = df_long["確認部品"].value_counts()
df_parts_count

確認部品
コイル        4
Buck-IC    3
MCU        2
異物         2
はんだ        1
コネクタ       1
NTF        1
Name: count, dtype: int64

In [44]:
#sell 7
df_cross = df_long.pivot_table(
    values="ticket_id",
    index="site",
    columns="確認部品",
    aggfunc="count",
    fill_value=0,
)
df_cross

確認部品,Buck-IC,MCU,NTF,はんだ,コイル,コネクタ,異物
site,,,,,,,
GSE,0,0,0,1,1,1,1
ISE,1,1,0,0,1,0,1
MELMB,1,0,0,0,2,0,0
WSE,1,1,1,0,0,0,0


In [45]:
#sell 8
df_cross["total"] = df_cross.sum(axis=1)
df_cross

確認部品,Buck-IC,MCU,NTF,はんだ,コイル,コネクタ,異物,total
site,,,,,,,,
GSE,0,0,0,1,1,1,1,4
ISE,1,1,0,0,1,0,1,4
MELMB,1,0,0,0,2,0,0,3
WSE,1,1,1,0,0,0,0,3


In [46]:
#sell9
df_cross_sort = df_cross.sort_values(
    "total",
    ascending=False,
)
df_cross_sort

確認部品,Buck-IC,MCU,NTF,はんだ,コイル,コネクタ,異物,total
site,,,,,,,,
GSE,0,0,0,1,1,1,1,4
ISE,1,1,0,0,1,0,1,4
MELMB,1,0,0,0,2,0,0,3
WSE,1,1,1,0,0,0,0,3


In [47]:
#cell 10
df_cross_sort = df_cross_sort.reset_index()
df_cross_sort


確認部品,site,Buck-IC,MCU,NTF,はんだ,コイル,コネクタ,異物,total
0,GSE,0,0,0,1,1,1,1,4
1,ISE,1,1,0,0,1,0,1,4
2,MELMB,1,0,0,0,2,0,0,3
3,WSE,1,1,1,0,0,0,0,3
